In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/q16_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 3 ###

### cell 3 (optimized for cudf)
# restrict part to only the columns and rows we need
sizes = [49, 14, 23, 45, 19, 3, 36, 9]
part_filtered = (
    part
    [(part.P_BRAND != "Brand#45")
     & (~part.P_TYPE.str.contains(r"^MEDIUM POLISHED"))
     & part.P_SIZE.isin(sizes)]
    [["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]]
)

# restrict partsupp to the columns we need
partsupp_filtered = partsupp[["PS_PARTKEY", "PS_SUPPKEY"]]

# inner‐join part and partsupp
total = (
    part_filtered
    .merge(partsupp_filtered,
           left_on="P_PARTKEY", right_on="PS_PARTKEY",
           how="inner")[["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]]
)

# build a deduplicated list of suppliers to exclude
supplier_keys = (
    supplier
    [supplier.S_COMMENT.str.contains(r"Customer(\S|\s)*Complaints")]
    ["S_SUPPKEY"].drop_duplicates()
)

# filter out any PS_SUPPKEY that appears in that supplier list
total = total[~total.PS_SUPPKEY.isin(supplier_keys)]

# group, count unique suppliers, rename, and sort
total = (
    total
    .groupby(["P_BRAND", "P_TYPE", "P_SIZE"],
              as_index=False, sort=False)
    .agg({"PS_SUPPKEY": "nunique"})
    .rename(columns={"PS_SUPPKEY": "SUPPLIER_CNT"})
    .sort_values(
        by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
        ascending=[False, True, True, True]
    )
)
